In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thnngvn/res-vi/Test.txt
/kaggle/input/datasets/thnngvn/res-vi/Train.txt
/kaggle/input/datasets/thnngvn/res-vi/Dev.txt
/kaggle/input/models/thnngvn/code-mtl/pytorch/default/1/train_mtl_acsa.py


In [2]:
!pip install pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 56.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.0 MB/s eta 0:00:00


In [3]:
!python /kaggle/input/models/thnngvn/code-mtl/pytorch/default/1/train_mtl_acsa.py \
      --train_path /kaggle/input/datasets/thnngvn/res-vi/Train.txt --dev_path /kaggle/input/datasets/thnngvn/res-vi/Dev.txt \
      --test_path /kaggle/input/datasets/thnngvn/res-vi/Test.txt \
      --model_name vinai/phobert-base-v2 \
      --loss_weighting gradnorm \
      --num_attention_heads 8 \
      --adapter_dim 256 \
      --dropout 0.10 \
      --epochs 20 \
      --batch_size 16 \
      --eval_batch_size 32 \
      --encoder_lr 5e-5 \
      --head_lr 1e-4 \
      --weight_decay 0.01 \
      --warmup_ratio 0.06 \
      --patience 5 \
      --lambda_acd 0.7 \
      --lambda_sent 1.3 \
      --lambda_joint 1.0 \
      --seed 42 \
      --output_dir outputs/phobert_mtl_acsa


[train] samples=7,028, avg_labels/sample=1.346
  sentiment: {'neutral': 2533, 'negative': 1746, 'positive': 5179}
  categories:
    AMBIENCE#GENERAL                815
    DRINKS#PRICES                   170
    DRINKS#QUALITY                  729
    DRINKS#STYLE&OPTIONS            465
    FOOD#PRICES                     401
    FOOD#QUALITY                   1995
    FOOD#STYLE&OPTIONS             1571
    LOCATION#GENERAL                373
    RESTAURANT#GENERAL              902
    RESTAURANT#MISCELLANEOUS        523
    RESTAURANT#PRICES               421
    SERVICE#GENERAL                1093

[dev] samples=771, avg_labels/sample=1.366
  sentiment: {'positive': 579, 'negative': 196, 'neutral': 278}
  categories:
    AMBIENCE#GENERAL                 91
    DRINKS#PRICES                    19
    DRINKS#QUALITY                   81
    DRINKS#STYLE&OPTIONS             52
    FOOD#PRICES                      45
    FOOD#QUALITY                    222
    FOOD#STYLE&OPTIONS       

In [4]:
import json
from pathlib import Path


def convert_prediction_json_to_txt(
    input_path: str,
    output_path: str,
):
    input_path = Path(input_path)
    output_path = Path(output_path)

    # =========================================================
    # Load JSON / JSONL
    # =========================================================
    if input_path.suffix.lower() == ".jsonl":
        samples = []

        with input_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                samples.append(json.loads(line))

    elif input_path.suffix.lower() == ".json":
        with input_path.open("r", encoding="utf-8") as f:
            samples = json.load(f)

        if isinstance(samples, dict):
            if "predictions" in samples:
                samples = samples["predictions"]
            elif "data" in samples:
                samples = samples["data"]
            else:
                raise ValueError(
                    "JSON phải là list hoặc chứa key 'predictions' / 'data'."
                )

    else:
        raise ValueError(
            f"Unsupported format: {input_path.suffix}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)

    # =========================================================
    # Convert
    # =========================================================
    with output_path.open("w", encoding="utf-8") as f:

        for idx, sample in enumerate(samples, start=1):

            sample_id = sample.get(
                "id",
                sample.get("sample_id", idx),
            )

            text = sample.get(
                "text",
                sample.get(
                    "raw_text",
                    sample.get("sentence", ""),
                ),
            )

            predictions = sample.get(
                "prediction",
                sample.get(
                    "predictions",
                    sample.get("pred", []),
                ),
            )

            # Tránh trường hợp prediction = None
            if predictions is None:
                predictions = []

            labels = []

            # =================================================
            # Parse predictions
            # =================================================
            for pred in predictions:

                # tránh lỗi nếu phần tử không phải dict
                if not isinstance(pred, dict):
                    continue

                category = pred.get(
                    "category",
                    pred.get("aspect"),
                )

                sentiment = pred.get(
                    "sentiment",
                    pred.get("polarity"),
                )

                if category is None or sentiment is None:
                    continue

                sentiment = str(sentiment).lower().strip()

                # Không ghi NONE
                if sentiment in {
                    "none",
                    "absent",
                    "not_present",
                    "",
                }:
                    continue

                labels.append(
                    f"{{{category}, {sentiment}}}"
                )

            # =================================================
            # FALLBACK:
            # Nếu prediction rỗng / parse xong không còn label
            # =================================================
            if not labels:
                labels.append(
                    "{RESTAURANT#GENERAL, neutral}"
                )

            # =================================================
            # Write format giống Test.txt
            # =================================================
            f.write(f"#{sample_id}\n")
            f.write(str(text).strip() + "\n")
            f.write(", ".join(labels) + "\n\n")

    print(f"Converted: {len(samples)} samples")
    print(f"Output: {output_path}")

In [5]:
convert_prediction_json_to_txt(
  input_path="outputs/phobert_mtl_acsa/test_predictions.jsonl",
  output_path="test_predictions.txt",
)

Converted: 1938 samples
Output: test_predictions.txt


In [6]:
import re
import sys

def get_labels_from_filename(filename):
    labels = []
    with open(filename, 'r', encoding = 'utf-8') as file:
        datasets = file.read()
        count = 0
        for line in datasets.split('\n'):
            if line != '':
                if count == 0:
                    count += 1
                elif count == 1:
                    count += 1
                elif count == 2:
                    labels.append(line.strip())
                    count = 0
        file.close()
    #print(len(labels))
    return labels

def clean_label(label):
    label = re.sub('[^A-Za-z#&]', '', label)
    label = re.sub('\\s+', ' ', label)
    return label

def convert_labels_to_dict(labels):
    dict_labels = []
    for label in labels:
        label_line = label.split('},')
        _dict = {}
        for objectLabel in label_line:
          try:
              aspect = clean_label(objectLabel.split(',')[0]).strip()
              polarity = clean_label(objectLabel.split(',')[1]).strip()
          except:
              print(label_line)
          _dict[aspect] = polarity
        dict_labels.append(_dict)
    return dict_labels

def get_common_attributeEntities(dict_labels):
    AttributeEntities = []
    for _dict in dict_labels:
        for key in _dict:
            if key not in AttributeEntities:
                AttributeEntities.append(key)
    AttributeEntities = sorted(AttributeEntities)
    return AttributeEntities

def get_aspects(dict_labels):
    aspects = []
    for _dict in dict_labels:
        for key in _dict:
            aspects.append(key)
    return aspects

def count_aspects(labels, Common_AttributeEntities):
    aspects = get_aspects(labels)
    num_aspects = [0] * len(Common_AttributeEntities)
    for aspect in aspects:
        num_aspects[Common_AttributeEntities.index(aspect)] += 1
    return num_aspects

def evaluation_labels(gold_labels, answer_labels, Common_AttributeEntities):
    num_aspect_gold = count_aspects(gold_labels, Common_AttributeEntities)
    num_aspect_answer = count_aspects(answer_labels, Common_AttributeEntities)
    correct_answer_aspects = [0] * len(Common_AttributeEntities)
    correct_answer_labels = [0] * len(Common_AttributeEntities)

    for i, _dict in enumerate(answer_labels):
        for key in _dict:
            if key in gold_labels[i].keys():
                correct_answer_aspects[Common_AttributeEntities.index(key)] += 1
                if answer_labels[i][key].strip() == gold_labels[i][key].strip():
                    correct_answer_labels[Common_AttributeEntities.index(key)] += 1
    #print('Correct Answer Aspects: ', correct_answer_aspects)
    #print('---------------------------------------------------')
    #print('Correct Answer Labels: ', correct_answer_labels)
    #print('---------------------------------------------------')
    #infor_evaluation(correct_answer_aspects, num_aspect_answer, num_aspect_gold, Common_AttributeEntities)
    #print('---------------------------------------------------')
    infor_evaluation(correct_answer_labels, num_aspect_answer, num_aspect_gold, Common_AttributeEntities)

def infor_evaluation(correct_answer, num_aspect_answer, num_aspect_gold, Common_AttributeEntities):
    for aspect in Common_AttributeEntities:
        if correct_answer[Common_AttributeEntities.index(aspect)] == 0:
            p = r = f = 0.0
        else:
            p = correct_answer[Common_AttributeEntities.index(aspect)] * 100 / num_aspect_answer[Common_AttributeEntities.index(aspect)]
            r = correct_answer[Common_AttributeEntities.index(aspect)] * 100 / num_aspect_gold[Common_AttributeEntities.index(aspect)]
            f = 2 * p * r / (p + r)
        print(aspect)
        print('%0.2f\t%0.2f\t%0.2f' % (p, r, f))
    p = sum(correct_answer) * 100 / sum(num_aspect_answer)
    r = sum(correct_answer) * 100 / sum(num_aspect_gold)
    f = 2 * p * r / (p + r)
    print('-------------------------------------------------------------')
    print('-------------------------------------------------------------')
    print('Mean Precision score: ', round(p,2))
    print('Mean Recall score: ', round(r,2))
    print('Mean F1 score: ', round(f,2))
    print('-------------------------------------------------------------')
    print('-------------------------------------------------------------')

def evaluation_system(gold_labels, answer_labels):
    gold_dicts = convert_labels_to_dict(gold_labels)
    answer_dicts = convert_labels_to_dict(answer_labels)
    AttributeEntities = get_common_attributeEntities(gold_dicts)
    #print('---------------INFORMATION FILE--------------------')
    #print('Aspect Name: ', AttributeEntities)
    #print("Aspect Gold: ", count_aspects(gold_dicts, AttributeEntities))
    #print("Aspect Answer: ", count_aspects(answer_dicts, AttributeEntities))
    #print('---------------------------------------------------')
    evaluation_labels(gold_dicts, answer_dicts, AttributeEntities)

def evaluation_system_by_file(file_gold, file_predict):
    gold_labels = get_labels_from_filename(file_gold)
    answer_labels = get_labels_from_filename(file_predict)
    evaluation_system(gold_labels, answer_labels)

In [7]:
evaluation_system_by_file("/kaggle/input/datasets/thnngvn/res-vi/Test.txt", "test_predictions.txt")

AMBIENCE#GENERAL
83.69	85.90	84.78
DRINKS#PRICES
77.08	78.72	77.89
DRINKS#QUALITY
76.73	76.35	76.54
DRINKS#STYLE&OPTIONS
66.13	63.57	64.82
FOOD#PRICES
81.74	83.93	82.82
FOOD#QUALITY
79.79	84.12	81.90
FOOD#STYLE&OPTIONS
73.18	73.68	73.43
LOCATION#GENERAL
73.64	77.88	75.70
RESTAURANT#GENERAL
63.04	69.32	66.03
RESTAURANT#MISCELLANEOUS
71.14	73.10	72.11
RESTAURANT#PRICES
79.46	76.07	77.73
SERVICE#GENERAL
85.11	86.80	85.95
-------------------------------------------------------------
-------------------------------------------------------------
Mean Precision score:  76.39
Mean Recall score:  78.51
Mean F1 score:  77.43
-------------------------------------------------------------
-------------------------------------------------------------
